# Capítulo 3 · Actividad: transformación, codificación y escalado

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/exercises/chapter_03/03_actividad_transformacion_escalado.ipynb)

Vas a aplicar lo que viste en la notebook de clase `05_transformacion_y_escalado` a **tres datasets distintos**, hasta dejar cada uno listo para un modelo.

### Antes de empezar

1. **Copia esta notebook a `workspace/`** y trabaja sobre la copia.
2. Si todavía no tienes los datos, descárgalos desde la raíz del repo:
   ```bash
   uv run python scripts/download_data.py
   ```

### Cómo trabajar

- Cada ejercicio tiene una celda `# Tu código aquí`. Añade todas las celdas que necesites.
- En las celdas ✍️ **justifica tus decisiones**.
- Al terminar, ejecuta **Kernel → Restart & Run All**.

| Parte | Datos | Tema |
|---|---|---|
| A. Propinas | `data/tips.csv` | Variables nuevas, codificación y escalado a mano |
| B. Pingüinos | `data/penguins.csv` | Codificación y escalado con scikit-learn |
| C. Viviendas de California | `data/california_housing.csv` | `Pipeline` completo |

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (FunctionTransformer, MinMaxScaler, OneHotEncoder,
                                   OrdinalEncoder, StandardScaler)

# Que scikit-learn devuelva DataFrames (con nombres de columna) en lugar de arrays de NumPy
set_config(transform_output="pandas")

# Carpeta data/ en la raíz del repo: subimos de carpeta en carpeta hasta dar con pyproject.toml
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"

---
## Parte A · Propinas 💵

Usaremos el dataset de propinas: `total_bill` (cuenta), `tip` (propina), `sex`, `smoker`, `day` (`Thur`, `Fri`, `Sat`, `Sun`), `time` (`Lunch`/`Dinner`) y `size` (personas en la mesa). **Queremos predecir la propina** (`tip`).

### Ejercicio 1 · Tipos de variables

Carga los datos en `propinas` y clasifica cada columna (salvo `tip`, que es el *target*) como **numérica**, **nominal**, **binaria** u **ordinal**.

In [ ]:
propinas = pd.read_csv(DATA_DIR / 'tips.csv')
propinas.head()

✍️ **Tu respuesta:** al revisar las columnas del dataset podemos ver que total_bill es numerica continua porque es dinero con centavos y size es numerica discreta porque cuenta personas en la mesa con numeros enteros. luego sex, smoker y time son binarias porque solo tienen dos opciones cada una como hombre o mujer, si o no, y comida o cena. por ultimo day es nominal porque son solo los nombres de los dias de la semana aunque tambien se podria ver como ordinal si los ordenamos por el calendario pero no tienen un valor matematico como tal.

### Ejercicio 2 · Variables nuevas

a) Crea la columna `cuenta_por_persona`.

b) Un compañero propone crear también `pct_propina` (la propina como porcentaje de la cuenta) para usarla en el modelo.

In [ ]:
propinas['cuenta_por_persona'] = propinas['total_bill'] / propinas['size']
propinas['pct_propina'] = propinas['tip'] / propinas['total_bill']
propinas.head()

✍️ **Pregunta:** ¿se puede usar `pct_propina` como variable para predecir la propina? ¿Por qué?

✍️ **Tu respuesta:** no se puede usar porque para sacar ese porcentaje necesitas conocer la propina desde antes y justamente la propina es lo que queremos predecir. si la metemos al modelo seria como hacer trampa o fuga de datos porque en la vida real cuando llega un cliente nuevo al restaurante todavia no sabemos cuanto va a dejar de propina.

### Ejercicio 3 · Variables binarias

Convierte `sex`, `smoker` y `time` en columnas de 0 y 1, guardando el resultado en columnas nuevas (`sex_num`, `smoker_num`, `time_num`).

In [ ]:
propinas['sex_num'] = propinas['sex'].map({'Female': 0, 'Male': 1})
propinas['smoker_num'] = propinas['smoker'].map({'No': 0, 'Yes': 1})
propinas['time_num'] = propinas['time'].map({'Lunch': 0, 'Dinner': 1})
propinas[['sex', 'sex_num', 'smoker', 'smoker_num', 'time', 'time_num']].head()

### Ejercicio 4 · El día: ¿one-hot u ordinal?

a) Codifica `day` con one-hot, con y sin `drop_first`.

b) Codifica `day` de forma ordinal respetando el orden de la semana.

In [ ]:
day_ohe = pd.get_dummies(propinas['day'], prefix='day', dtype=int)
day_ohe_drop = pd.get_dummies(propinas['day'], prefix='day', drop_first=True, dtype=int)

orden_dias = {'Thur': 0, 'Fri': 1, 'Sat': 2, 'Sun': 3}
propinas['day_ordinal'] = propinas['day'].map(orden_dias)

display(day_ohe.head(3))
display(day_ohe_drop.head(3))
display(propinas[['day', 'day_ordinal']].head(3))

✍️ **Pregunta:** ¿qué codificación elegirías para `day` y por qué? ¿Y en `drop_first=True`, qué día se eliminó y cómo se reconoce una fila de ese día?

✍️ **Tu respuesta:** para day yo elegiria one-hot porque los dias de la semana son categorias independientes y no hay una jerarquia matematica donde un dia valga mas que otro. con drop_first=True la columna que se elimino fue friday que es el primer dia en orden alfabetico, y una fila de ese dia se reconoce facilmente porque tiene valor 0 en todas las demas columnas de los dias.

### Ejercicio 5 · Escalado a mano

Escala `total_bill` con **min-max** y con **z-score**, sin usar scikit-learn. Comprueba con `describe()` la media, la desviación, el mínimo y el máximo de cada versión.

In [ ]:
min_val = propinas['total_bill'].min()
max_val = propinas['total_bill'].max()
propinas['total_bill_minmax'] = (propinas['total_bill'] - min_val) / (max_val - min_val)

mean_val = propinas['total_bill'].mean()
std_val = propinas['total_bill'].std()
propinas['total_bill_zscore'] = (propinas['total_bill'] - mean_val) / std_val

propinas[['total_bill', 'total_bill_minmax', 'total_bill_zscore']].describe()

### Ejercicio 6 · Logaritmo

a) Calcula la asimetría de `total_bill` y de `np.log1p(total_bill)`.

b) Dibuja sus dos histogramas, uno al lado del otro.

In [ ]:
skew_orig = propinas['total_bill'].skew()
log_bill = np.log1p(propinas['total_bill'])
skew_log = log_bill.skew()
print('asimetria original:', skew_orig)
print('asimetria log1p:', skew_log)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(propinas['total_bill'], bins=20, edgecolor='black')
axes[0].set_title('total_bill original')
axes[1].hist(log_bill, bins=20, edgecolor='black')
axes[1].set_title('log1p(total_bill)')
plt.tight_layout()
plt.show()

✍️ **Pregunta:** si escalaras `total_bill` con z-score en lugar de aplicarle el logaritmo, ¿cambiaría su asimetría? ¿Por qué?

✍️ **Tu respuesta:** no cambiaria la asimetria porque el escalado z-score es una transformacion lineal donde solo le restas la media y divides entre la desviacion estandar entonces la forma de la grafica y su colita quedan exactamente iguales solo cambian los numeros del eje. en cambio el logaritmo si cambia la asimetria porque es una transformacion no lineal que comprime los valores mas grandes y hace que los datos queden mas parejos y simetricos.

---
## Parte B · Pingüinos 🐧 (con scikit-learn)

Queremos predecir la **especie** de un pingüino a partir de sus medidas, la isla y el sexo.

### Ejercicio 7 · Preparar y separar

a) Carga `penguins.csv` en `pinguinos` y elimina las filas con valores faltantes.

b) Separa `X` (todas las columnas salvo `species`) e `y` (`species`).

c) Divide en train y test (20% para test, `random_state=42`). ¿Cuántas filas tiene cada parte?

In [ ]:
pinguinos = pd.read_csv(DATA_DIR / 'penguins.csv').dropna()
X = pinguinos.drop(columns='species')
y = pinguinos['species']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('filas en train:', len(X_train))
print('filas en test:', len(X_test))

### Ejercicio 8 · `OneHotEncoder`

a) Ajusta un `OneHotEncoder` con la columna `island` de **train** y muestra las categorías que aprendió.

b) Transforma este DataFrame con dos pingüinos nuevos: uno de `Dream` y otro de la isla `Anvers`.
```python
nuevos = pd.DataFrame({"island": ["Dream", "Anvers"]})
```

In [ ]:
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(X_train[['island']])
print('categorias aprendidas:', ohe.categories_)

nuevos = pd.DataFrame({'island': ['Dream', 'Anvers']})
display(ohe.transform(nuevos))

✍️ **Pregunta:** ¿qué pasó con el pingüino de `Anvers`? ¿Qué habría pasado sin `handle_unknown="ignore"`?

✍️ **Tu respuesta:** con el pinguino de anvers lo que paso fue que todas sus columnas quedaron en cero porque esa isla no venia en los datos de entrenamiento con los que se ajusto el encoder. si no le hubieramos puesto handle_unknown="ignore" el programa se habria detenido y lanzado un error diciendo que encontro una categoria desconocida que nunca habia visto.

### Ejercicio 9 · `OrdinalEncoder`

a) Crea en `X_train` una columna de texto `tamaño` agrupando `body_mass_g` en `"pequeño"` (hasta 3500 g), `"mediano"` (hasta 4500 g) y `"grande"` (más de 4500 g).

b) Codifícala con un `OrdinalEncoder` **sin indicar** el orden y muestra las categorías que aprendió.

c) Vuelve a codificarla indicando el orden correcto.

In [ ]:
def categorizar_masa(m):
    if m <= 3500:
        return 'pequeño'
    elif m <= 4500:
        return 'mediano'
    else:
        return 'grande'

X_train = X_train.copy()
X_train['tamaño'] = X_train['body_mass_g'].apply(categorizar_masa)

oe_sin_orden = OrdinalEncoder()
oe_sin_orden.fit(X_train[['tamaño']])
print('categorias sin orden:', oe_sin_orden.categories_)
display(oe_sin_orden.transform(X_train[['tamaño']]).head(3))

orden = [['pequeño', 'mediano', 'grande']]
oe_con_orden = OrdinalEncoder(categories=orden)
oe_con_orden.fit(X_train[['tamaño']])
print('categorias con orden correcto:', oe_con_orden.categories_)
display(oe_con_orden.transform(X_train[['tamaño']]).head(3))

✍️ **Pregunta:** ¿qué número recibe cada tamaño en b)? ¿Por qué es un problema?

✍️ **Tu respuesta:** en el inciso b al no ponerle el orden scikit-learn acomoda las palabras por orden alfabetico y le asigna 0 a grande, 1 a mediano y 2 a pequeño. esto es un problema grave porque le estariamos diciendo al modelo que grande vale menos que mediano y que pequeño es el mas grande de todos, invirtiendo la logica real de los tamaños y confundiendo el aprendizaje del modelo.

### Ejercicio 10 · `StandardScaler`

Estandariza las cuatro medidas numéricas (`bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`) de la forma correcta y compara la media de cada columna en train y en test.

In [ ]:
num_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train[num_cols])
X_test_scaled = scaler.transform(X_test[num_cols])

print('medias en train:')
print(X_train_scaled.mean())
print('\nmedias en test:')
print(X_test_scaled.mean())

✍️ **Pregunta:** ¿por qué la media en test no es exactamente 0? ¿Es un error?

✍️ **Tu respuesta:** la media en test no da exactamente cero porque el escalador se ajusto unicamente con los datos de train usando su promedio y su desviacion estandar. no es ningun error sino que es lo correcto porque test simula datos nuevos que el modelo no conoce y si usaramos los datos de test para ajustar el escalador estariamos cometiendo fuga de datos.

### Ejercicio 11 · `ColumnTransformer`

Construye un `ColumnTransformer` que:
- estandarice las cuatro medidas numéricas,
- aplique one-hot a `island` y `sex` (con una sola columna para `sex`).

Ajústalo con train, transforma train y test, y muestra la forma y las primeras filas del resultado.

In [ ]:
num_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

preprocesador = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('island', OneHotEncoder(drop=None, sparse_output=False), ['island']),
        ('sex', OneHotEncoder(drop='first', sparse_output=False), ['sex']),
    ]
)

X_train_prep = preprocesador.fit_transform(X_train)
X_test_prep = preprocesador.transform(X_test)

print('forma train:', X_train_prep.shape)
print('forma test:', X_test_prep.shape)
display(X_train_prep.head())

### Ejercicio 12 · Un pingüino nuevo

Prepara con tu `preprocesador` este pingüino recién medido:
```python
nuevo = pd.DataFrame([{"island": "Biscoe", "bill_length_mm": 47.5, "bill_depth_mm": 15.0,
                       "flipper_length_mm": 218.0, "body_mass_g": 5200.0, "sex": "Male"}])
```

In [ ]:
nuevo = pd.DataFrame([{
    'island': 'Biscoe',
    'bill_length_mm': 47.5,
    'bill_depth_mm': 15.0,
    'flipper_length_mm': 218.0,
    'body_mass_g': 5200.0,
    'sex': 'Male'
}])

nuevo_prep = preprocesador.transform(nuevo)
display(nuevo_prep)

---
## Parte C · Viviendas de California 🏠 (`Pipeline` completo)

Queremos predecir el valor de las viviendas (`MedHouseVal`) de cada distrito a partir del resto de columnas: `MedInc`, `HouseAge`, `AveRooms`, `AveBedrms`, `Population`, `AveOccup`, `Latitude` y `Longitude`.

### Ejercicio 13 · ¿Qué variables transformar?

Carga `california_housing.csv` en `california` y calcula la asimetría de cada variable (salvo el *target*). Después, calcula la asimetría que tendrían las variables más asimétricas si les aplicaras `np.log1p`.

In [ ]:
california = pd.read_csv(DATA_DIR / 'california_housing.csv')
features = california.drop(columns='MedHouseVal')
skewness = features.skew()
print('asimetria original:')
print(skewness.sort_values(ascending=False))

asimetricas = ['AveRooms', 'AveBedrms', 'Population', 'AveOccup']
skew_log = np.log1p(features[asimetricas]).skew()
print('\nasimetria aplicando log1p:')
print(skew_log)

✍️ **Pregunta:** ¿a qué variables les aplicarías el logaritmo? ¿Funciona igual de bien en todas?

✍️ **Tu respuesta:** yo le aplicaria el logaritmo a las variables que tienen una asimetria muy grande con colas largas como averooms, avebedrms, population y aveoccup para reducir el impacto de los valores extremos y acercarlas a una distribucion normal. no funciona igual de bien en todas porque por ejemplo en latitude y longitude no tiene ningun sentido aplicarlo al ser coordenadas geograficas, y en algunas que tienen valores atipicos gigantescos la asimetria baja mucho pero todavia pueden quedar colas largas.

### Ejercicio 14 · El `Pipeline` completo

a) Separa `X` e `y`, y divide en train y test (20% para test, `random_state=42`).

b) Construye un preprocesador que aplique **logaritmo y después estandarización** a las variables asimétricas que elegiste, y **solo estandarización** al resto.

c) Ajústalo con train, transforma train y test, y comprueba que las medias en train son 0 y las desviaciones 1.

In [ ]:
X = california.drop(columns='MedHouseVal')
y = california['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

skewed_cols = ['AveRooms', 'AveBedrms', 'Population', 'AveOccup']
other_cols = [c for c in X.columns if c not in skewed_cols]

log_pipeline = Pipeline([
    ('log', FunctionTransformer(np.log1p)),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('log_scale', log_pipeline, skewed_cols),
    ('scale', StandardScaler(), other_cols)
])

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print('medias en train:')
print(X_train_prep.mean().round(4))
print('\ndesviaciones en train:')
print(X_train_prep.std().round(4))